# Azure 函数调用示例

本 notebook 展示如何在 Azure OpenAI 服务中使用函数调用（function calling）功能。函数允许 chat completions 的调用方定义模型可以使用的能力，从而将其
功能扩展到外部工具和数据源。

你可以在 OpenAI 的博客上阅读更多关于聊天函数的内容：https://openai.com/blog/function-calling-and-other-api-updates

**注意**：聊天函数需要 gpt-4 和 gpt-35-turbo 的 `-0613` 标签及以后的模型版本。旧版本的模型不支持该功能。

## 设置

首先，安装必要的依赖并导入将要使用的库。

In [ ]:
! pip install "openai>=1.0.0,<2.0.0"
! pip install python-dotenv

In [ ]:
import os
import openai
import dotenv

dotenv.load_dotenv()

### 身份验证

Azure OpenAI 服务支持多种身份验证机制，包括 API 密钥和 Azure Active Directory 令牌凭据。

In [2]:
use_azure_active_directory = False  # Set this flag to True if you are using Azure Active Directory

#### 使用 API 密钥进行身份验证

要让 OpenAI SDK 使用 *Azure API Key*，我们需要将 `api_key` 设置为与端点关联的密钥（你可以在 [Azure Portal](https://portal.azure.com) 的 *"Resource Management"* 下的 *"Keys and Endpoints"* 中找到该密钥）。你也可以在这里找到资源的端点。

In [3]:
if not use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]

    client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2023-09-01-preview"
    )

#### 使用 Azure Active Directory 进行身份验证
接下来看看如何通过 Azure Active Directory 进行身份验证。我们首先安装 `azure-identity` 库。该库将提供身份验证所需的令牌凭据，并通过 `get_bearer_token_provider` 辅助函数帮助我们构建令牌凭据提供程序。建议使用 `get_bearer_token_provider`，而不是向 `AzureOpenAI` 提供静态令牌，因为该 API 会自动为你缓存和刷新令牌。

关于如何为 Azure OpenAI 设置 Azure Active Directory 身份验证的更多信息，请参阅[文档](https://learn.microsoft.com/azure/ai-services/openai/how-to/managed-identity)。

In [ ]:
! pip install "azure-identity>=1.15.0"

In [5]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

if use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]

    client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        azure_ad_token_provider=get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"),
        api_version="2023-09-01-preview"
    )

> 注意：如果未提供以下参数，AzureOpenAI 会从对应的环境变量中推断它们：

- `api_key` 来自 `AZURE_OPENAI_API_KEY`
- `azure_ad_token` 来自 `AZURE_OPENAI_AD_TOKEN`
- `api_version` 来自 `OPENAI_API_VERSION`
- `azure_endpoint` 来自 `AZURE_OPENAI_ENDPOINT`


## 部署

在本节中，我们将创建一个可用于调用函数的 GPT 模型部署。

### 部署：在 Azure OpenAI Studio 中创建
让我们部署一个用于 chat completions 的模型。前往 https://portal.azure.com, 找到你的 Azure OpenAI 资源，然后进入 Azure OpenAI Studio。点击 "Deployments" 选项卡，然后为你想用于 chat completions 的模型创建一个部署。你为模型指定的部署名称将在下面的代码中使用。

In [4]:
deployment = "" # Fill in the deployment name from the portal here

## 函数

完成设置和身份验证后，你现在可以在 Azure OpenAI 服务中使用函数了。这将分为以下几个步骤：

1. 定义函数
2. 将函数定义传入 chat completions API
3. 使用响应中的参数调用函数
4. 将函数的响应回传给 chat completions API

#### 1. 定义函数

可以定义一个函数列表，每个函数包含函数名称、可选的描述，以及函数接受的参数（用 JSON schema 描述）。

In [5]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "format": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit to use. Infer this from the users location.",
                },
            },
            "required": ["location"],
        },
    }
]

#### 2. 将函数定义传入 chat completions API

现在我们可以将函数传入 chat completions API。如果模型判断应该调用该函数，choice 上的 `finish_reason` 会被设置为 "tool_calls"，要调用哪个函数及其参数的详细信息会出现在 `message` 中。你也可以设置 `tool_choice` 关键字参数，强制模型调用某个特定函数（例如 `{"type": "function", "function": {"name": get_current_weather}}`）。默认情况下它被设置为 `auto`，由模型自行选择是否调用函数。 

In [ ]:
messages = [
    {"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."},
    {"role": "user", "content": "What's the weather like today in Seattle?"}
]

chat_completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    tools=functions,
)
print(chat_completion)

#### 3. 使用响应中的参数调用函数

函数调用的名称将是最初提供的名称之一，参数则是与函数定义中所含 schema 相匹配的 JSON。

In [ ]:
import json

def get_current_weather(request):
    """
    This function is for illustrative purposes.
    The location and unit should be used to determine weather
    instead of returning a hardcoded response.
    """
    location = request.get("location")
    unit = request.get("unit")
    return {"temperature": "22", "unit": "celsius", "description": "Sunny"}

function_call = chat_completion.choices[0].message.tool_calls[0].function
print(function_call.name)
print(function_call.arguments)

if function_call.name == "get_current_weather":
    response = get_current_weather(json.loads(function_call.arguments))

#### 4. 将函数的响应回传给 chat completions API

函数的响应应序列化为一条新消息，并将其 role 设置为 "function"。现在模型会利用这些响应数据来组织它的回答。

In [ ]:
messages.append(
    {
        "role": "function",
        "name": "get_current_weather",
        "content": json.dumps(response)
    }
)

function_completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    tools=functions,
)

print(function_completion.choices[0].message.content.strip())